# StarLayer User Guide (Notebook)

This notebook is located at https://github.com/hidden-graph/starlayer/blob/main/docs/user-guide-v1.ipynb

## How to run this notebook

1. pip install from the github repository.
2. Run cells from top to bottom so shared variables remain available.


In [ ]:
pip install "git+https://github.com/hidden-graph/starlayer.git"

  Cloning https://github.com/hidden-graph/starlayer.git to /tmp/pip-req-build-qd9sapem
  Running command git clone --filter=blob:none --quiet https://github.com/hidden-graph/starlayer.git /tmp/pip-req-build-qd9sapem
  Resolved https://github.com/hidden-graph/starlayer.git to commit 14c985ff90db5e41a6dc6091747b4ea4b0abec6c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 7.5 MB/s eta 0:00:00
  Created wheel for starlayer: filename=starlayer-0.1.0-py3-none-any.whl size=389140 sha256=e61ec0d0a23aea57a1ca1e89937dd12236424e7d28ff8d160f63ba661185dce4
  Stored in directory: /tmp/pip-ephem-wheel-cache-ejeahc_y/wheels/2b

In [ ]:
from starlayergraph import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics
Extension of the rdflib graph model to support RDF 1.2.  
- Triple terms and statement resources
- Reification via rdf:reifies and statement metadata
- Direction-tagged strings such as "hello"@en--ltr and "مرحبا"@ar--rtl

In [26]:
#create the graph, and assign namespace
g = StarLayerGraph()
g.bind("ex", EX)

#create a triple term
tt = TripleTerm(EX.bob, EX.knows, EX.carol)

#create a reifer (EX.clain) associated with triple term and add to graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.wikipedia))

print((EX.claim, RDF.reifies, tt) in g)
print(g.serialize(format="turtle12"))

True
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)

#add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.bob, EX.knows, EX.carol)))
g.add((EX.other, RDF.reifies, (EX.bob, EX.likes, EX.dana)))

#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.bob):
    print(t)
print(g.has_triple_term(EX.bob, EX.knows, EX.carol))
print(g.has_triple_term(EX.bob, EX.knows, EX.dana))

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
True
False


In [ ]:
#RDF.reifies is the common approach to making a statement about a statement, RDF 1.2 allows triple terms in object position of any triple.

#add reifiers to the graph
g.add((EX.dana, EX.said, (EX.bob, EX.knows, EX.carol)))

#triples accepts triple term as object to select triples
selectTriples = g.triples((None, None, (EX.bob, EX.knows, EX.carol)))

#qname_term is a starlayer function that adds qname transformation to triple terms as well.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))


ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>>
ex:dana ex:said <<( ex:bob ex:knows ex:carol )>>


### Finding what's been said about a statement
`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier ↔ triple-term ↔ annotation relationship directly, without writing SPARQL.

In [ ]:
# continues using g from the previous cell
g.add((EX.claim, EX.source, EX.wikipedia))

tt1 = (EX.bob, EX.knows, EX.carol)

# reifiers(): which reifier node(s) reify a given triple term?
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): which triple terms have at least one reifier?
for tt in g.reifications():
    print(tt)

# reifier_annotations(): a reifier's own annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# remove_reification(): undo just the rdf:reifies triple; annotations survive
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt1) in g)
print((EX.claim, EX.source, EX.wikipedia) in g)

['ex:claim']
<<( ex:bob ex:knows ex:carol )>>
<<( ex:bob ex:likes ex:dana )>>
ex:claim ex:source ex:wikipedia
<<( ex:bob ex:knows ex:carol )>>
False
True


### Set the direction of a string literal.

DirLangString has been added to set the direction of a string literal.


In [ ]:
g = StarLayerGraph()
g.bind("ex", EX)

#literals can include language direction.
g.add((EX.title, EX.value, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.title, EX.value, Literal("hello","en")))
g.add((EX.title, EX.value, DirLangString("hello","en","ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:title ex:value "hello"@en, "مرحبا"@ar--rtl, "hello"@en--ltr .



## 2. Parsing and Serialization
Supports parsing and serialization of RDF 1.2 content across the full set of rdflib supported formats. (turtle12 and longturtle12 for Turtle, nt12 and nq12 for N-Triples and N-Quads, trig12 and trix12 for datasets, rdfxml12,  and jsonld12. (jsonld does not have a published RDF 1.2 spec.)
- Quoted triple-term content
- Turtle annotation syntax
- Language-direction literals

In [ ]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
# "a triple as the subject of another statement" (N3-style { }, e.g. { ex:bob ex:knows ex:carol } ex:says ex:dana)
# is NOT supported here - it parses without error but silently produces zero triples, so don't reach for it.
# RDF 1.2 never allows a triple term as a subject at all (only object position); the reification shorthand
# section below already covers the real way to write this: << ex:bob ex:knows ex:carol >> ex:says ex:dana .
# parse mixed RDF 1.2 content: direction-tagged literals, quoted triples, and multiple reification forms
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals
    ex:note_en ex:text "hello"@en--ltr .
    ex:note_ar ex:text "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:source ex:wikipedia ;
      ex:confidence "high" .

    # anonymous inline annotation block
    ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} .

    # named reifier with annotations
    ex:bob ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} .

    # named reifier without annotation block
    ex:bob ex:worksWith ex:frank ~ ex:stmt2 .

    # an additional  quoted triple term reused in querie examples.
    ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .
''', format='turtle12')

print(g_parsed.serialize(format='turtle12'))
#other options:
#turtle12, longturtle12, nt12, nq12, trig12, trix12, rdfxml12,jsonld12

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:mentions <<( ex:bob ex:likes ex:dana )>> .

ex:bob ex:likes ex:dana {| ex:since "2020" ; ex:source ex:LinkedIn |} ;
    ex:mentions ex:erin ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:WikiData |} ;
    ex:worksWith ex:frank ~ ex:stmt2 .

ex:claim ex:confidence "high" ;
    ex:source ex:wikipedia ;
    rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

ex:note_ar ex:text "مرحبا"@ar--rtl .

ex:note_en ex:text "hello"@en--ltr .



## 3. SPARQL and query semantics
Supports SPARQL 1.2 query execution.
- Query over reified quoted triples
- Uses Turtle 1.2 quoted-triple syntax (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

Following examples use the graph parsed in section 2 above.

In [ ]:
# bind all terms inside a quoted triple and include statement metadata
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:knows)
}
ORDER BY ?claim ?s ?o
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        g_parsed.qname(row.source),
    )

ex:claim ex:bob ex:knows ex:carol ex:wikipedia


In [ ]:
# selective term binding inside a quoted triple pattern
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?person ?friend WHERE {
  ?claim rdf:reifies <<( ?person ex:knows ?friend )>> .
  FILTER(?friend = ex:carol)
}
ORDER BY ?claim ?person
""")

for row in rows:
    print(
        g_parsed.qname(row.claim),
        g_parsed.qname(row.person),
        g_parsed.qname(row.friend),
    )

ex:claim ex:bob ex:carol


In [32]:
# detect triple-term values dynamically with isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

for row in rows:
    print(g_parsed.qname(row.claim),g_parsed.qname(row.p))


ex:alice ex:mentions
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies
rr:0 rdf:reifies


### Additional SPARQL 1.2 functions
`TRIPLE(s, p, o)` is the function-call spelling of `<<( s p o )>>`. `SUBJECT()`/`PREDICATE()`/`OBJECT()` pull the three components back out of a bound triple term. `LANGDIR()`/`hasLANGDIR()`/`STRLANGDIR()` work with base direction directly; `LANG()`/`hasLANG()` are the ordinary rdflib functions, extended to also recognize a direction-tagged literal.

In [33]:
# TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:bob, ex:knows, ex:carol))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

ex:bob ex:knows ex:carol


In [34]:
# LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over the direction-tagged notes
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:text ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# STRLANGDIR() constructs a direction-tagged literal directly from plain strings
rows = g_parsed.query('SELECT ?lit WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

ex:note_ar ar rtl true
ex:note_en en ltr true
"hi"@en--ltr


### TTL annotation shorthand inside SPARQL queries
The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` forms used to *parse* `g_parsed` above (see the Turtle 1.2 content in the parsing section) also work directly inside a SPARQL WHERE clause — you query the shorthand the same way you wrote it, without expanding to `rdf:reifies`/`<<( )>>` by hand.

In [35]:
# {| ?pred ?val |}: query an anonymous reifier's annotations inline
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:bob ex:likes ex:dana {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# ~ ?r: bind the reifier itself for a named reifier
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:bob ex:worksWith ex:frank ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# << s p o >> ?pred ?val: reification shorthand, no assertion required
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:bob ex:likes ex:dana >> ?pred ?val
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

ex:since 2020
ex:source http://example.org/LinkedIn
ex:stmt2
ex:since 2020
ex:source http://example.org/LinkedIn


## 4. SHACL validation and rules

- Validation over RDF 1.2 graphs
- SHACL rule support
- Direction-aware datatype constraints
- Fine-grained severity and messages.
- New severity levels

### Is a value a triple term? `sh:nodeKind ( sh:TripleTerm )`
The list-valued form of `sh:nodeKind` recognizes `sh:TripleTerm` as one of its choices, letting a shape require that a value — e.g. the object of `rdf:reifies` — is genuinely an RDF 1.2 triple term, not a plain URI, blank node, or literal. Only the list form works; a bare `sh:nodeKind sh:TripleTerm` (no list) silently falls through to plain pySHACL's own check, which has no concept of triple terms at all.

In [36]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    [] a sh:NodeShape ;
      sh:targetSubjectsOf rdf:reifies ;
      sh:property [ sh:path rdf:reifies ; sh:nodeKind ( sh:TripleTerm ) ] .
""", format="turtle")

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .
""", format="turtle12")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms (real triple term):", result.conforms)

bad_data = StarLayerGraph()
bad_data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies ex:not_a_triple_term .
""", format="turtle")
result_bad = StarShaclValidator().validate(data_graph=bad_data, shacl_graph=shapes, meta_shacl=False)
print("conforms (plain URI, not a triple term):", result_bad.conforms)

ReportableRuntimeError: SHACL File does not validate against the SHACL Shapes SHACL (MetaSHACL) file.
Validation Report
Conforms: False
Results (1):
Constraint Violation in OrConstraintComponent (http://www.w3.org/ns/shacl#OrConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:or ( [ sh:in ( sh:BlankNode sh:IRI sh:Literal sh:BlankNodeOrIRI sh:BlankNodeOrLiteral sh:IRIOrLiteral ) ] [ sh:node shsh:ListShape ; sh:property [ sh:path [ sh:zeroOrMorePath rdf:rest ] ; sh:property [ sh:in ( sh:BlankNode sh:IRI sh:Literal sh:BlankNodeOrIRI sh:BlankNodeOrLiteral sh:IRIOrLiteral ) ; sh:path rdf:first ] ] ] ) ; sh:path sh:nodeKind ]
	Focus Node: [ sh:nodeKind ( sh:TripleTerm ) ; sh:path rdf:reifies ]
	Value Node: ( sh:TripleTerm )
	Result Path: sh:nodeKind
	Message: Node ( sh:TripleTerm ) must conform to one or more shapes in [ sh:in ( sh:BlankNode sh:IRI sh:Literal sh:BlankNodeOrIRI sh:BlankNodeOrLiteral sh:IRIOrLiteral ) ] , [ sh:node

### New severity levels: `sh:Debug`, `sh:Trace`
SHACL 1.2 adds two severities below `sh:Info`/`sh:Warning`. Unlike those two (which need `allow_warnings=True`/`allow_infos=True` to stop blocking `conforms`), a `sh:Debug`/`sh:Trace` result **never** blocks conformance — it's recorded in the report but has no effect on `result.conforms`, useful for constraints you want visibility into without failing validation over.

In [37]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
""", format="turtle")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:minCount 1 ; sh:severity sh:Debug ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print("conforms despite missing ex:age:", result.conforms)  # True - sh:Debug never blocks
print(result.report_text)

conforms despite missing ex:age: True
Validation Report
Conforms: False
Results (1):
Validation Result in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Debug
	Source Shape: [ sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ex:age ; sh:severity sh:Debug ]
	Focus Node: ex:alice
	Result Path: ex:age
	Message: Less than 1 values on ex:alice->ex:age



### Fine-grained severity via reification: `{| sh:severity ... |}`
A single constraint-value triple can carry its own `sh:severity`/`sh:deactivated` override via RDF 1.2's inline annotation shorthand — distinct from, and finer-grained than, a shape's own `sh:severity`. Currently wired for `sh:datatype`/`sh:uniqueMembers`/`sh:reificationRequired`/`sh:singleLine` (severity) and `sh:property` (deactivation), and only on a constraint declared directly on the targeted shape (not yet inside a nested `sh:property [...]` blank node).

In [38]:
SH = Namespace("http://www.w3.org/ns/shacl#")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AgeMustBeIntShape a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:datatype xsd:integer {| sh:severity sh:Warning |} .
""", format="turtle12")
data = StarLayerGraph()
data.add((EX.alice, EX.dummy, EX.alice))  # ex:alice is a URI, not an xsd:integer literal - violates

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
severities = {result.report_graph.qname(o) for _, _, o in
              result.report_graph.triples((None, SH.resultSeverity, None))}
print("severity from the annotation:", severities)  # {'sh:Warning'}, not the default sh:Violation
print("conforms:", result.conforms)                  # still False - Warning still blocks unless allowed

result2 = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False, allow_warnings=True)
print("conforms with allow_warnings=True:", result2.conforms)

severity from the annotation: {'sh:Warning'}
conforms: False
conforms with allow_warnings=True: True


In [39]:
#SHACL validation of a direction-tagged literal.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:n1 a ex:Note ;
      ex:label "hello"@en--ltr .
""", format="turtle12")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:LangShape a sh:NodeShape ;
      sh:targetClass ex:Note ;
      sh:property [ sh:path ex:label ; sh:datatype rdf:dirLangString ] .
""", format="turtle")
result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes)
print(result.conforms)

True


In [40]:
#NOTE - can we make the output clear by outputting the new triple.

#SHACL validation targets a reifier node (sh:targetSubjectsOf ex:confidence)
# and conditionally infers a new triple from its annotation - SHACL rules operating on RDF 1.2
# reification structure.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> ;
      ex:confidence "high" .
""", format="turtle12")
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ConfidenceRule a sh:NodeShape ;
    #nodes with high confidence should be trusted.
      sh:targetSubjectsOf ex:confidence ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ;
        sh:predicate ex:trusted ;
        sh:object ex:yes ;
        sh:condition [ sh:property [ sh:path ex:confidence ; sh:hasValue "high" ] ] ;
      ] .
""", format="turtle")
result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print((EX.claim, EX.trusted, EX.yes) in result.data_graph)
print(result.conforms)

True
True


## 5. Backend graph-store and format support

This section is about *where the data actually lives* — which store StarLayer talks to, and whether it speaks RDF 1.1 or native RDF 1.2 to get there. (RDF 1.2 format support — turtle12, nt12, nq12, trig12, trix12, rdfxml12, jsonld12, longturtle12 — is covered separately below, in "All eight RDF 1.2 formats".)

- In-memory backend (default) and native RDF 1.2 backend modes
- Dual-mode operation: the same store can be driven in RDF 1.1 (encoding, rewritten queries) or native RDF 1.2 mode
- Any rdflib `Store` plugin works transparently under the default RDF 1.1 backend — not just the built-in in-memory store

### Oxigraph — native RDF 1.2

Oxigraph 0.5.9+ speaks SPARQL 1.2 (triple-term syntax) directly over HTTP. Point `StarLayerGraph` at it with `backend='rdf-1.2'` and a `SPARQLUpdateStore`, and triple terms/direction-tagged literals go over the wire in their real syntax — no `tt:HASH` encoding, no query rewriting. **Illustrative only** — this needs a running Oxigraph instance (`docker run -d -p 7878:7878 ghcr.io/oxigraph/oxigraph serve --location /data --bind 0.0.0.0:7878`); not executed in this notebook.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

store = SPARQLUpdateStore(
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)
g = StarLayerGraph(store=store, identifier=EX.main, backend="rdf-1.2")
g.bind("ex", EX)
g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

rows = g.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g.qname(row.stmt))
```

### Fuseki — dual-mode: RDF 1.1 and RDF 1.2 against the same store

Fuseki 5.5+ also speaks native RDF 1.2. The dual-mode story: the *same* store connection works in either backend mode — omit `backend=` (defaults to `'rdf-1.1'`) to have StarLayer encode triple terms and rewrite SPARQL 1.2 syntax down to plain SPARQL 1.1 before sending it, for compatibility with any SPARQL 1.1-only endpoint; pass `backend='rdf-1.2'` to send native RDF 1.2 syntax directly once you know the endpoint supports it. **Illustrative only** — needs a running Fuseki instance (`docker run -d -p 3030:3030 atomgraph/fuseki:latest --update --mem --ping /starlayergraph`, Fuseki 5.5+ required for the RDF 1.2 mode); not executed here.

```python
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def make_store():
    return SPARQLUpdateStore(
        query_endpoint="http://localhost:3030/starlayergraph/query",
        update_endpoint="http://localhost:3030/starlayergraph/update",
        auth=("admin", "admin"),
    )

# RDF 1.1 mode: triple terms encoded, SPARQL 1.2 rewritten to 1.1 before sending
g11 = StarLayerGraph(store=make_store(), identifier=EX.main)   # backend='rdf-1.1' is the default
g11.bind("ex", EX)
g11.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))

# RDF 1.2 mode: native triple-term syntax sent directly, no rewriting
g12 = StarLayerGraph(store=make_store(), identifier=EX.main, backend="rdf-1.2")
g12.bind("ex", EX)
rows = g12.query("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:bob ex:knows ex:carol )>> }
""")
for row in rows:
    print(g12.qname(row.stmt))
```

### SQLAlchemy — RDF 1.1, real SQL-backed persistence

Unlike Oxigraph/Fuseki above, this one *is* executed in this notebook: `StarLayerGraph` is an ordinary `rdflib.Graph` subclass, so any rdflib `Store` plugin works transparently under the default RDF 1.1 (encoding) backend — including a real SQL database via `rdflib-sqlalchemy`. Needs the `sqlalchemy` extra (`pip install rdflib-sqlalchemy`); no native RDF 1.2 mode exists for this store (`rdflib-sqlalchemy` has no SPARQL 1.2 support of its own), so this is RDF 1.1-only.

In [41]:
!pip install -q rdflib-sqlalchemy

import tempfile
import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.qname_term(next(reader.triple_terms(subject=EX.bob))))
reader.close()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 37.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 18.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.
<<( ex:bob ex:knows ex:carol )>>
